## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://private-user-images.githubusercontent.com/181020547/465824799-12a2371b-8be2-4219-9b48-90503eb43c69.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NjAwNDgyMzcsIm5iZiI6MTc2MDA0NzkzNywicGF0aCI6Ii8xODEwMjA1NDcvNDY1ODI0Nzk5LTEyYTIzNzFiLThiZTItNDIxOS05YjQ4LTkwNTAzZWI0M2M2OS5wbmc_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1BS0lBVkNPRFlMU0E1M1BRSzRaQSUyRjIwMjUxMDA5JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwOVQyMjEyMTdaJlgtQW16LUV4cGlyZXM9MzAwJlgtQW16LVNpZ25hdHVyZT1iYTRmYTAzYjkzYjA2MGE4ZTZlYjQ4ODU1OWIwY2VlZWU0Mzk0YzdmMjQ1YTlhMDMyNmI3NWNlZTQxNDdlZGViJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCJ9.a8477QD1J4Lrmys7jB8gt_H5pdiKBsKsu3npEqZjEpo)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

#### ✅ Answer

**Part 1: Interrelationships between the three states**

- The three states work hierarchically: AgentState (main) passes research briefs to SupervisorState (manager), which delegates specific topics to ResearcherState (individual researchers). Data flows down (topics) and back up (findings). Multiple ResearcherStates can run in parallel, each focused on different aspects of the research question.

**Part 2: Why not a single huge state?**

- A single state would be like one person trying to do everything in a research organization which is inefficient. Separate states allow parallel researchers to work independently, and work more efficiently. It's also easier to scale up by adding more researchers without restructuring the entire system.

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

#### ✅ Answer

**Advantages:**

• Code split into logical modules for easier navigation  
• Components can be used in other projects without copying  
• Fix bugs in one file instead of hunting through notebook  
• Each module can be unit tested independently  
• Smaller files easier to track in git  
• Pre-compiled library imports faster

**Disadvantages:**

• Need to ensure library is properly installed  
• May need to jump between notebook and library files  
• New users need to understand both structures  
• Must package library with notebook  
• Notebook requires external files to run

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [16]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [17]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [19]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with analyzing the NBER working paper "How People Use ChatGPT." I understand you want insights about: (1) main findings about how people are using AI, (2) most common use cases, and (3) trends or patterns from the data. The PDF content you've provided contains comprehensive research data from the study covering ChatGPT usage from November 2022 through July 2025. I will now analyze this document and provide detailed insights based on the research findings.

Node: write_research_brief

Research Brief Generated:
I have a PDF document of the NBER working paper "How People Use ChatGPT" by Aaron Chatterji, Thomas Cunningham, David J. Deming, Zoe Hitzig, Christopher Ong, Carl Yan Shan, and Kevin Wadman (Working Paper 34255, September 2025). I need you to analyze this document and provide comprehensive insights about: (1) What are the main findings about how people are using AI (specifically ChatG

# How People Use ChatGPT: Comprehensive Analysis of the NBER Research

## Main Findings About How People Use AI (ChatGPT)

The NBER working paper "How People Use ChatGPT" reveals unprecedented patterns in AI adoption and usage that fundamentally reshape our understanding of how artificial intelligence is being integrated into daily life. By July 2025, ChatGPT reached more than 700 million weekly active users—representing approximately 10% of the global adult population—with the platform processing roughly 18 billion messages per week or 2.5 billion messages per day (equivalent to more than 30,000 messages per second) [1][3].

The most striking finding is the dramatic shift from work-related to personal use. Non-work messages have grown from 53% in mid-2024 to over 70% in mid-2025, representing a fundamental transformation in how AI chatbots are being utilized [1][10]. This shift occurred primarily through changing usage patterns within existing user cohorts rather than simply new user composition, suggesting that people are discovering new ways to integrate AI into their personal lives as they become more familiar with the technology.

The researchers developed a novel taxonomy classifying user intent into three categories that provides crucial insights into human-AI interaction patterns. "Asking" activities account for approximately 49% of messages, where users seek information or advice for decision-making. "Doing" activities represent about 40% of messages, involving requests for ChatGPT to perform tasks or create output. "Expressing" activities comprise roughly 11% of messages, where users engage with the AI for purposes that don't involve seeking information or requesting specific tasks [1][4]. Notably, in work contexts, "Doing" activities increase to approximately 56% and focus largely on writing tasks like editing, summarizing, and translating.

The economic implications are profound. The authors conclude that ChatGPT's strongest economic value emerges as a decision-support tool, particularly important in knowledge-intensive jobs [1][3]. Rather than simply automating tasks, the platform's primary value lies in helping people make decisions, generate text, and find clarity quickly. Most work-related usage centers on obtaining and interpreting information, and making decisions or solving problems creatively, highlighting AI's role as an intellectual augmentation tool rather than replacement.

User satisfaction metrics reveal high engagement quality, with positive interactions outnumbering negative ones by approximately 4:1 [1]. This suggests that users are generally finding value in their AI interactions across diverse use cases.

## Most Common Use Cases Identified in the Research

The research identifies three dominant use cases that collectively account for nearly 80% of all ChatGPT conversations: Practical Guidance, Seeking Information, and Writing [1][3][6].

**Practical Guidance** emerges as the single most common use case at 29% of all usage, remaining remarkably stable throughout the study period [4][9]. This category encompasses personalized tutoring and advice, how-to guidance across various topics, and creative ideation. The consistency of this usage pattern suggests a fundamental human need for AI-powered guidance and problem-solving assistance that spans both personal and professional contexts.

**Seeking Information** accounts for 24% of usage and showed significant growth from 14% to 24% over the study period [7][9]. This category appears to function as a very close substitute for traditional web search, indicating that users are increasingly turning to conversational AI for information discovery rather than search engines. The growth in this category suggests users appreciate the interactive, contextual nature of AI-powered information retrieval compared to traditional search methodologies.

**Writing** represents 24% of usage but notably declined from 36% of all usage in July 2024 to 24% a year later [7][9]. Despite this relative decline, writing remains crucial, especially in work contexts where it accounts for 40% of work-related messages [10]. Importantly, about two-thirds of all writing messages involve ChatGPT modifying existing user text (editing, critiquing, translating, summarizing) rather than creating entirely new content from scratch. This pattern highlights the AI's role as a sophisticated editing and enhancement tool rather than simply a content generation engine.

Several lower-volume but significant use cases provide additional insights into AI adoption patterns. Education accounts for 10% of all messages [7], while computer programming represents only about 4.2% of messages—much lower than expected given the extensive developer-focused media coverage [1][4][7]. This finding challenges common assumptions about AI's primary applications and suggests broader, more diverse adoption than typically portrayed in technology discourse.

Personal and social use cases remain relatively niche, with relationships or personal reflection accounting for 1.9% of messages and games/roleplay representing just 0.4% of usage [4][7]. These patterns suggest that while ChatGPT has become a mainstream tool, its primary value remains functional rather than social or entertainment-focused.

## Trends and Patterns from Data Collected (November 2022 - July 2025)

The growth trajectory of ChatGPT represents one of the most dramatic technology adoption patterns in recorded history. ChatGPT reached 1 million users just 5 days after launch in November 2022, making it the fastest-growing application in history until Threads briefly surpassed it [5]. The platform hit 100 million weekly active users in early November 2023—less than one year after launch—with the user base doubling every 7-8 months since then [3].

**Usage Intensity and Engagement Patterns**

Message volume has grown even faster than user adoption, with volume increasing 5.8x in the last year while users grew 3.2x [3]. Daily message volume expanded by more than 5x between July 2024 and July 2025 [10]. Critically, this growth represents increased usage within existing user cohorts rather than just new user acquisition, with all cohorts of users substantially increasing their daily message volume beginning in early 2025. This pattern suggests that ChatGPT became significantly better or more user-friendly during this period, encouraging deeper engagement from existing users [3].

**Demographic Evolution and Global Adoption**

The demographic composition of ChatGPT users has undergone remarkable transformation, particularly regarding gender distribution. Initially, early adopters were predominantly male, with ChatGPT having 80% male users at launch [3][4]. However, the gender gap has narrowed dramatically, with male users dropping to 48% by June 2025 [10]. By July 2025, 52% of active users had typically female first names, suggesting that the gender gap in ChatGPT usage may have closed completely [3]. Among users with names that could be classified as either masculine or feminine, the share with typically feminine names grew from 37% in January 2024 to 52% in July 2025 [5][7].

Age demographics reveal significant youth adoption, with nearly half of all messages coming from users under 26, and over 45% of users being under 25 years old [1][5]. This pattern suggests that AI chatbots are becoming integral tools for digital natives entering their professional and academic years.

**Geographic and Economic Development Patterns**

Global adoption patterns reveal fascinating insights about AI accessibility and economic development. Higher growth rates have been observed in lower-income countries compared to wealthier nations [3]. Usage rates have converged across countries regardless of GDP per capita, with middle-income countries showing 5-6x growth compared to 3x in the richest countries [3]. Global adoption is accelerating fastest in low-to-middle income countries with GDP per capita between $10,000-40,000 [4], with growth in the lowest-income countries being four times higher than in the highest-income countries [7].

This geographic distribution pattern suggests that AI chatbots may be serving as democratizing technologies, providing access to sophisticated information processing and decision-support capabilities regardless of traditional economic barriers. The rapid adoption in developing economies could represent a significant leap-frogging opportunity, similar to mobile phone adoption patterns in the early 2000s.

**Comparative Historical Context**

To contextualize ChatGPT's growth, the research provides compelling historical comparisons. ChatGPT is growing much faster than Google search did historically—Google took 8 years to reach 1 billion daily searches, while ChatGPT reached 1 billion daily messages in less than 2 years [3]. This acceleration suggests fundamental changes in how quickly transformative technologies can achieve global scale in the current digital infrastructure environment.

The evolution from work-focused to personal-focused usage represents perhaps the most significant trend identified in the research. This shift indicates that as users become more familiar with AI capabilities, they discover applications beyond their initial professional needs, integrating AI into broader aspects of their daily lives. This pattern suggests we are still in the early stages of understanding AI's ultimate role in human society and productivity.

### Sources

[1] How People Really Use ChatGPT: Findings from NBER Research: https://techmaniacs.com/2025/09/15/how-people-really-use-chatgpt-findings-from-nber-research/

[2] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255

[3] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt

[4] New OpenAI Study Reveals How 700 Million People Actually Use...: https://www.reddit.com/r/OpenAI/comments/1niaw9p/new_openai_study_reveals_how_700_million_people/

[5] Number of ChatGPT Users (October 2025) - Exploding Topics: https://explodingtopics.com/blog/chatgpt-users

[6] ChatGPT Study: 1 In 4 Conversations Now Seek Information: https://www.searchenginejournal.com/chatgpt-study-1-in-4-conversations-now-seek-information/556104/

[7] OpenAI Shares Data on How People Are Using ChatGPT: https://www.socialmediatoday.com/news/openai-shares-data-on-how-people-are-using-chatgpt-ai-chatbot/760193/

[8] OpenAI Says 70% of ChatGPT Use is Non-Work Related: https://www.pymnts.com/artificial-intelligence-2/2025/openai-70percent-chatgpt-use-non-work-related/

[9] [PDF] How People Use ChatGPT - National Bureau of Economic Research: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf

[10] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

#### ✅ Answer

In [ ]:
# Activity #1: Configuration Experiments - Running Research with Different Settings
print("🔬 Running Configuration Experiments - Observing Research Differences")

# Create a simple test research request
test_request = "What are the latest trends in AI research? Focus on machine learning and natural language processing."

# Experiment 1: Increase Parallelism
print("\n1️⃣ Testing Increased Parallelism (10 concurrent researchers)")
config_parallel = {
    "configurable": {
        **config["configurable"],  # Copy existing config
        "max_concurrent_research_units": 10,  # More researchers working simultaneously
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3
    }
}

# Experiment 2: Deeper Research
print("\n2️⃣ Testing Deeper Research (8 iterations, 15 tool calls)")
config_deeper = {
    "configurable": {
        **config["configurable"],  # Copy existing config
        "max_researcher_iterations": 8,   # Supervisor can delegate more times
        "max_react_tool_calls": 15      # Each researcher can search more
    }
}

# Experiment 3: Use Anthropic Native Search
print("\n3️⃣ Testing Anthropic Native Search")
config_anthropic = {
    "configurable": {
        **config["configurable"],  # Copy existing config
        "search_api": "anthropic"  # Use Claude's built-in web search
    }
}

# Experiment 4: Disable Clarification
print("\n4️⃣ Testing Disabled Clarification")
config_no_clarify = {
    "configurable": {
        **config["configurable"],  # Copy existing config
        "allow_clarification": False  # Skip clarification phase
    }
}

# Function to run experiments and capture results
async def run_experiment(exp_name, exp_config):
    print(f"\n--- {exp_name} Results ---")
    results = {
        "notes_count": 0,
        "report_length": 0,
        "nodes_executed": [],
        "final_report": "",
        "start_time": None,
        "end_time": None
    }
    
    import time
    results["start_time"] = time.time()
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": test_request}]},
        exp_config,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            results["nodes_executed"].append(node_name)
            
            if node_name == "supervisor_tools" and "notes" in node_output:
                results["notes_count"] = len(node_output['notes'])
                print(f"Research notes collected: {results['notes_count']}")
                
            elif node_name == "final_report_generation" and "final_report" in node_output:
                results["report_length"] = len(node_output['final_report'])
                results["final_report"] = node_output['final_report']
                print(f"Final report length: {results['report_length']} characters")
                print(f"Report preview: {node_output['final_report'][:200]}...")
    
    results["end_time"] = time.time()
    results["duration"] = results["end_time"] - results["start_time"]
    
    return results

# Run Experiment 1: Increased Parallelism
print("Running Experiment 1...")
parallel_results = await run_experiment("Increased Parallelism", config_parallel)

print(f"\n📊 Experiment 1 Summary:")
print(f"- Duration: {parallel_results['duration']:.2f} seconds")
print(f"- Nodes executed: {len(set(parallel_results['nodes_executed']))}")
print(f"- Research notes: {parallel_results['notes_count']}")
print(f"- Report length: {parallel_results['report_length']} characters")

# Run Experiment 2: Deeper Research
print("\n" + "="*50)
print("Running Experiment 2...")
deeper_results = await run_experiment("Deeper Research", config_deeper)

print(f"\n📊 Experiment 2 Summary:")
print(f"- Duration: {deeper_results['duration']:.2f} seconds")
print(f"- Nodes executed: {len(set(deeper_results['nodes_executed']))}")
print(f"- Research notes: {deeper_results['notes_count']}")
print(f"- Report length: {deeper_results['report_length']} characters")

# Run Experiment 3: Anthropic Native Search
print("\n" + "="*50)
print("Running Experiment 3...")
anthropic_results = await run_experiment("Anthropic Native Search", config_anthropic)

print(f"\n📊 Experiment 3 Summary:")
print(f"- Duration: {anthropic_results['duration']:.2f} seconds")
print(f"- Nodes executed: {len(set(anthropic_results['nodes_executed']))}")
print(f"- Research notes: {anthropic_results['notes_count']}")
print(f"- Report length: {anthropic_results['report_length']} characters")

# Run Experiment 4: Disable Clarification
print("\n" + "="*50)
print("Running Experiment 4...")
no_clarify_results = await run_experiment("Disabled Clarification", config_no_clarify)

print(f"\n📊 Experiment 4 Summary:")
print(f"- Duration: {no_clarify_results['duration']:.2f} seconds")
print(f"- Nodes executed: {len(set(no_clarify_results['nodes_executed']))}")
print(f"- Research notes: {no_clarify_results['notes_count']}")
print(f"- Report length: {no_clarify_results['report_length']} characters")

# Final Comparison
print("\n" + "="*60)
print("🔍 FINAL COMPARISON OF ALL EXPERIMENTS")
print("="*60)

experiments = [
    ("Increased Parallelism", parallel_results),
    ("Deeper Research", deeper_results), 
    ("Anthropic Native Search", anthropic_results),
    ("Disabled Clarification", no_clarify_results)
]

for name, results in experiments:
    print(f"\n{name}:")
    print(f"  ⏱️  Duration: {results['duration']:.2f}s")
    print(f"  📝 Notes: {results['notes_count']}")
    print(f"  📄 Report Length: {results['report_length']} chars")
    print(f"  🔄 Nodes: {len(set(results['nodes_executed']))}")



## Experiment Analysis / Conclusion

In [26]:
# Extract Experiment Results from Previous Output and Create Table
import pandas as pd
import re

# Extract data from the experiment results that were already run
# We'll parse the output that was generated

# From the actual experiment output, let's extract the key metrics
experiment_results = []

# Parse the results from the previous cell output
# Experiment 1: Increased Parallelism
exp1_data = {
    "Experiment": "Experiment 1",
    "Configuration": "Increased Parallelism (10 researchers)",
    "Duration": 6.98,
    "Nodes_Executed": 1,
    "Notes_Collected": 0,
    "Report_Length": 0,
    "Status": "Partial Success"
}

# Experiment 2: Deeper Research  
exp2_data = {
    "Experiment": "Experiment 2",
    "Configuration": "Deeper Research (8 iterations, 15 tool calls)",
    "Duration": 9.33,
    "Nodes_Executed": 1,
    "Notes_Collected": 0,
    "Report_Length": 0,
    "Status": "Partial Success"
}

# Experiment 3: Anthropic Native Search
exp3_data = {
    "Experiment": "Experiment 3", 
    "Configuration": "Anthropic Native Search",
    "Duration": 7.99,
    "Nodes_Executed": 1,
    "Notes_Collected": 0,
    "Report_Length": 0,
    "Status": "Partial Success"
}

# Experiment 4: Disabled Clarification
exp4_data = {
    "Experiment": "Experiment 4",
    "Configuration": "Disabled Clarification", 
    "Duration": 124.90,
    "Nodes_Executed": 4,
    "Notes_Collected": 0,
    "Report_Length": 627,
    "Status": "Failed (Rate Limit)"
}

# Combine all results
all_results = [exp1_data, exp2_data, exp3_data, exp4_data]

# Create DataFrame from actual results
df_results = pd.DataFrame(all_results)

# Display the table
print("📊 Experiment Results Table (From Actual Output)")
print("=" * 80)
print(df_results.to_string(index=False))

# Status breakdown
status_counts = df_results['Status'].value_counts()
print(f"\n📈 Status Breakdown:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")


📊 Experiment Results Table (From Actual Output)
  Experiment                                 Configuration  Duration  Nodes_Executed  Notes_Collected  Report_Length              Status
Experiment 1        Increased Parallelism (10 researchers)      6.98               1                0              0     Partial Success
Experiment 2 Deeper Research (8 iterations, 15 tool calls)      9.33               1                0              0     Partial Success
Experiment 3                       Anthropic Native Search      7.99               1                0              0     Partial Success
Experiment 4                        Disabled Clarification    124.90               4                0            627 Failed (Rate Limit)

📈 Status Breakdown:
  Partial Success: 3
  Failed (Rate Limit): 1


## Comparision Result

Because some of my experiments with changed configurations failes with several reasons, I really can't compare the full results. But with this experimet, I can definitely conclude as below for sure.

- Performance degradation with changed settings
- Configuration stability is crucial for successful execution

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs